# Rerank ViDoRe v3 HR text

This notebook compares **No Reranker** (BM25 candidate order) with the locally configured `RERANK_MODEL` on the same post-retrieval text. An optional path uses Nemotron 3 Embed to retrieve candidates, then measures its own before-and-after reranking quality. It uses **all pages, queries, and relevance judgments in [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr)**—one dataset within the larger ViDoRe v3 collection, not the entire collection. The dataset card lists 1,110 pages and 1,908 queries.

**Plan for a long run.** With 12 candidates per query, the main HR evaluation scores about 22,896 query–page pairs locally. Enabling embedding retrieval doubles the reranked pairs and adds embedding inference for all 1,110 pages and 1,908 queries. The first run downloads the reranker weights, plus the embedding model if enabled. This is a **text-only worked comparison**, not an official visual ViDoRe benchmark score: it uses capped extracted markdown, not page images. Reranking cannot recover a relevant page absent from its candidate list.

## Configuration and dependencies

This notebook runs the reranker selected by `RERANK_MODEL` locally with a BF16-capable CUDA GPU and enough memory for model weights plus inference overhead. The model ID in the configuration cell is the swap point for Nemotron 3.5 Rerank when it is released. A one-line swap assumes its released `AutoProcessor` retains the same cross-encoder input method; verify the released model card first. If your environment requires authenticated downloads, export `HF_TOKEN` before starting Jupyter; for a token in a local `hf-token` file, run `export HF_TOKEN="$(< /path/to/hf-token)"` with the file's actual path. Never paste or commit a token. The loader uses model-provided Python code (`trust_remote_code=True`); review that code before running it in a sensitive environment. Only extracted page markdown is sent to the model; no page images are decoded or reranked. Loading the full HR corpus still downloads roughly 442 MB of source data on the first run because the dataset stores images in its corpus files. Allow time and disk space for both downloads.

The optional Jupyter magic below installs into the selected kernel. Use `transformers>=5.16.1` as a shared baseline for this model-switching example; recheck requirements when changing `RERANK_MODEL`. The optional [Nemotron 3 Embed model](https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16) also requires `sentence-transformers>=5.4.1`. Install a CUDA-enabled PyTorch build matched to your driver using the [PyTorch selector](https://pytorch.org/get-started/locally/). Both local paths use PyTorch SDPA, so FlashAttention is not required; the reranker does not need Sentence Transformers. Restart the kernel after installing packages. The dataset card identifies the pinned revision used for end-to-end evaluation.

In [1]:
# %pip install --upgrade "datasets>=4.8.2"
# After installing a matching CUDA-enabled PyTorch build:
# %pip install --upgrade "transformers>=5.16.1,<6"
# Optional embedding retrieval:
# %pip install --upgrade "sentence-transformers>=5.4.1"

from importlib.util import find_spec

missing = [name for name in ("datasets",) if find_spec(name) is None]
if missing:
    raise ImportError(
        f"Missing {', '.join(missing)} in this kernel. Run the optional %pip command above, "
        "restart the kernel, and run from the top."
    )

DATASET_ID = "vidore/vidore_v3_hr"
DATASET_REVISION = "95f2f83a5a09590a89e34960479f9438e48bca77"
RUN_EMBEDDING_RETRIEVAL = False  # Extra full-HR pass; keep the BM25 path first
CANDIDATES_PER_QUERY = 12
TOP_K = 5
DISPLAY_QUERY_ROWS = 10
MAX_PASSAGE_CHARS = 4000
RERANK_BATCH_SIZE = 1
RERANK_MAX_LENGTH = 512
RERANK_MODEL = "nvidia/llama-nemotron-rerank-vl-1b-v2"
RERANK_LABEL = RERANK_MODEL.rsplit("/", 1)[-1]
RERANK_METRIC = f"{RERANK_LABEL} nDCG"
EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16"
EMBED_BATCH_SIZE = 2
assert CANDIDATES_PER_QUERY >= TOP_K > 0
assert DISPLAY_QUERY_ROWS > 0 and MAX_PASSAGE_CHARS > 0
assert RERANK_BATCH_SIZE > 0 and RERANK_MAX_LENGTH > 0
assert EMBED_BATCH_SIZE > 0

local_missing = [name for name in ("torch", "transformers")
                 if find_spec(name) is None]
if local_missing:
    raise ImportError("Local reranking requires " + ", ".join(local_missing) +
                      ". Use the optional install guidance above.")
from importlib.metadata import version
from packaging.version import Version
if Version(version("transformers")) < Version("5.16.1"):
    raise RuntimeError("This notebook requires transformers>=5.16.1 for the configured reranker.")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Local reranking needs a CUDA GPU visible to this kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The BF16 reranker needs a CUDA GPU with BF16 support.")
print(f"Reranker model card: https://huggingface.co/{RERANK_MODEL}")

Reranker model card: https://huggingface.co/nvidia/llama-nemotron-rerank-vl-1b-v2


## Load all HR pages, queries, and judgments

ViDoRe v3 HR supplies page images, extracted markdown, query text, and graded qrels (1 = critically relevant, 2 = fully relevant). We load the entire HR test split but select only corpus IDs and markdown for BM25 and reranking, so no page images are decoded. Unjudged pages are treated as relevance 0. This still evaluates only HR, not the other ViDoRe v3 datasets.

In [2]:
from datasets import load_dataset

def load_split(config):
    return load_dataset(
        DATASET_ID, config, split="test", revision=DATASET_REVISION,
    )

corpus_text = load_split("corpus").select_columns(["corpus_id", "markdown"])
pages = [
    {"corpus_id": row["corpus_id"], "markdown": row["markdown"] or ""}
    for row in corpus_text
]
queries = sorted(load_split("queries"), key=lambda row: row["query_id"])
page_by_id = {row["corpus_id"]: row for row in pages}
relevance_by_query = {}
for row in load_split("qrels"):
    if row["corpus_id"] not in page_by_id:
        raise ValueError("A qrel refers to a page outside the HR corpus.")
    judgments = relevance_by_query.setdefault(row["query_id"], {})
    page_id = row["corpus_id"]
    judgments[page_id] = max(judgments.get(page_id, 0), row["score"])

print(f"Loaded all {len(pages)} HR pages, {len(queries)} queries, and "
      f"judgments for {len(relevance_by_query)} queries.")
if len(pages) < CANDIDATES_PER_QUERY:
    raise ValueError("CANDIDATES_PER_QUERY exceeds the HR corpus size.")
if any(row["query_id"] not in relevance_by_query for row in queries):
    raise ValueError("Some HR queries have no relevance judgments.")

/home/edwardk/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded all 1110 HR pages, 1908 queries, and judgments for 1908 queries.


## Produce post-retrieval candidates

BM25 is the first-stage retriever here. It searches **every HR page for every HR query**, then returns the top CANDIDATES_PER_QUERY pages to either option. In an application, replace its output with chunks or pages from your existing retriever and keep the reranking cells unchanged. Queries with no relevant page among those candidates remain in the evaluation: a reranker cannot fix that retrieval miss.

In [3]:
import math
import re
from collections import Counter

def tokens(text):
    return re.findall(r"[\w]+", text.lower())

page_tokens = [tokens(row["markdown"]) for row in pages]
term_counts = [Counter(row) for row in page_tokens]
document_frequency = Counter(term for row in term_counts for term in row)
average_length = sum(map(len, page_tokens)) / len(page_tokens)

def retrieve(query, depth=CANDIDATES_PER_QUERY):
    query_terms = set(tokens(query))
    scores = []
    for row, counts, words in zip(pages, term_counts, page_tokens):
        score = 0.0
        for term in query_terms:
            frequency = counts[term]
            if not frequency:
                continue
            df = document_frequency[term]
            idf = math.log(1 + (len(pages) - df + 0.5) / (df + 0.5))
            score += idf * frequency * 2.2 / (
                frequency + 1.2 * (0.25 + 0.75 * len(words) / average_length)
            )
        scores.append((score, row["corpus_id"]))
    best = sorted(scores, key=lambda pair: (-pair[0], pair[1]))[:depth]
    return [
        {"corpus_id": page_id, "retrieval_score": score, "original_rank": rank}
        for rank, (score, page_id) in enumerate(best, start=1)
    ]

examples = [
    {"query_id": row["query_id"], "query": row["query"],
     "candidates": retrieve(row["query"]),
     "relevance": relevance_by_query[row["query_id"]]}
    for row in queries
]
zero_recall = sum(
    not any(hit["corpus_id"] in example["relevance"]
            for hit in example["candidates"])
    for example in examples
)
print(f"Prepared {len(examples)} HR queries with "
      f"{CANDIDATES_PER_QUERY} BM25 candidates each.")
print(f"{zero_recall} queries have no judged-relevant page in their candidate set.")

Prepared 1908 HR queries with 12 BM25 candidates each.
1331 queries have no judged-relevant page in their candidate set.


## Optional: retrieve candidates with Nemotron 3 Embed

Set `RUN_EMBEDDING_RETRIEVAL=True` above to replace BM25's candidate generator for a second, separate comparison. This path embeds the **same capped page markdown passed to the reranker** and all HR queries with [Nemotron 3 Embed 1B BF16](https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16), retrieves the top `CANDIDATES_PER_QUERY` pages by cosine similarity, then lets the reranker score those candidates. BM25 still indexes full page markdown, so a cross-source difference includes both retrieval method and input length. Different candidate sets can have different recall; compare each reranked result to its **own** retrieval baseline.

The embedder runs before the configured reranker is loaded and is released afterward to limit peak GPU memory. If you change the flag after a run, restart the kernel and run the notebook from the top. This optional path adds a model download, embedding inference, and another full reranking pass.

In [4]:
embedding_examples = []
if RUN_EMBEDDING_RETRIEVAL:
    if find_spec("sentence_transformers") is None:
        raise ImportError("Run the optional Sentence Transformers %pip command, then restart the kernel.")
    if Version(version("sentence-transformers")) < Version("5.4.1"):
        raise RuntimeError("Nemotron 3 Embed requires sentence-transformers>=5.4.1.")
    import gc
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(
        EMBED_MODEL, device="cuda",
        model_kwargs={"dtype": torch.bfloat16, "attn_implementation": "sdpa"},
    )
    documents = [page["markdown"][:MAX_PASSAGE_CHARS] or " " for page in pages]
    document_vectors = embedder.encode_document(
        documents, batch_size=EMBED_BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=True,
    )
    query_vectors = embedder.encode_query(
        [row["query"] for row in queries], batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True, show_progress_bar=True,
    )
    similarities = query_vectors @ document_vectors.T
    for query_index, row in enumerate(queries):
        best = sorted(
            range(len(pages)),
            key=lambda page_index: (
                -float(similarities[query_index, page_index]),
                str(pages[page_index]["corpus_id"]),
            ),
        )[:CANDIDATES_PER_QUERY]
        candidates = [
            {"corpus_id": pages[page_index]["corpus_id"],
             "retrieval_score": float(similarities[query_index, page_index]),
             "original_rank": rank}
            for rank, page_index in enumerate(best, start=1)
        ]
        embedding_examples.append(
            {"query_id": row["query_id"], "query": row["query"],
             "candidates": candidates,
             "relevance": relevance_by_query[row["query_id"]]}
        )
    del embedder, documents, document_vectors, query_vectors, similarities
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Prepared embedding candidates for all {len(embedding_examples)} HR queries.")
else:
    print("Embedding retrieval skipped; set RUN_EMBEDDING_RETRIEVAL=True to enable it.")

Embedding retrieval skipped; set RUN_EMBEDDING_RETRIEVAL=True to enable it.


## Rerank the same candidates with the configured model

The local reranker scores the **same text-only candidates** as the BM25 baseline. Its checkpoint exposes a cross-encoder processor that takes a question, `doc_text`, and an empty `doc_image`. Extracted markdown is capped to keep inputs manageable; changing that cap or the token limit can change results. No page images are sent to the model.

If `HF_TOKEN` is set, the model loader uses it for download. No token is required for a public download, and query and passage text stay on your machine. The next cell starts the full run without prompting. An interrupted run can resume within the same kernel without repeating completed queries.

In [5]:
import os
import time

hf_token = os.environ.get("HF_TOKEN") or None
from transformers import AutoModelForSequenceClassification, AutoProcessor

local_processor = AutoProcessor.from_pretrained(
    RERANK_MODEL, token=hf_token, trust_remote_code=True,
    rerank_max_length=RERANK_MAX_LENGTH,
)
local_reranker = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL, token=hf_token, trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
).eval().to("cuda")

def local_rerank(query, candidates):
    scores = []
    for start in range(0, len(candidates), RERANK_BATCH_SIZE):
        batch = candidates[start : start + RERANK_BATCH_SIZE]
        pairs = [
            {"question": query,
             "doc_text": page_by_id[hit["corpus_id"]]["markdown"][:MAX_PASSAGE_CHARS] or " ",
             "doc_image": ""}
            for hit in batch
        ]
        inputs = local_processor.process_queries_documents_crossencoder(pairs)
        inputs = {
            name: value.to("cuda") if isinstance(value, torch.Tensor) else value
            for name, value in inputs.items()
        }
        with torch.inference_mode():
            logits = local_reranker(**inputs, return_dict=True).logits
        scores.extend(float(score) for score in logits.view(-1).cpu().tolist())
    if len(scores) != len(candidates) or any(not math.isfinite(score) for score in scores):
        raise ValueError("Local reranker must return one finite score per candidate.")
    reranked = [dict(hit, rerank_score=score) for hit, score in zip(candidates, scores)]
    return sorted(reranked, key=lambda hit: hit["rerank_score"], reverse=True)

rerank_candidates = local_rerank

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████████████| 601/601 [00:00<00:00, 3287.79it/s]


In [6]:
pending = [example for example in examples if "reranked" not in example]
print(f"Pending: {len(pending)} local reranking queries, up to "
      f"{len(pending) * CANDIDATES_PER_QUERY} query–page pairs.")
if pending:
    for completed, example in enumerate(pending, start=1):
        started = time.perf_counter()
        example["reranked"] = rerank_candidates(example["query"], example["candidates"])
        example["rerank_seconds"] = time.perf_counter() - started
        if completed % 100 == 0 or completed == len(pending):
            print(f"Completed {completed}/{len(pending)} pending queries.")
print(f"Reranked {len(examples)} HR queries locally with {RERANK_MODEL}.")

Pending: 1908 local reranking queries, up to 22896 query–page pairs.
Completed 100/1908 pending queries.
Completed 200/1908 pending queries.
Completed 300/1908 pending queries.
Completed 400/1908 pending queries.
Completed 500/1908 pending queries.
Completed 600/1908 pending queries.
Completed 700/1908 pending queries.
Completed 800/1908 pending queries.
Completed 900/1908 pending queries.
Completed 1000/1908 pending queries.
Completed 1100/1908 pending queries.
Completed 1200/1908 pending queries.
Completed 1300/1908 pending queries.
Completed 1400/1908 pending queries.
Completed 1500/1908 pending queries.
Completed 1600/1908 pending queries.
Completed 1700/1908 pending queries.
Completed 1800/1908 pending queries.
Completed 1900/1908 pending queries.
Completed 1908/1908 pending queries.
Reranked 1908 HR queries locally with nvidia/llama-nemotron-rerank-vl-1b-v2.


In [7]:
if RUN_EMBEDDING_RETRIEVAL:
    pending_embeddings = [example for example in embedding_examples
                          if "reranked" not in example]
    print(f"Pending: {len(pending_embeddings)} embedding-candidate reranking queries, "
          f"up to {len(pending_embeddings) * CANDIDATES_PER_QUERY} query–page pairs.")
    for completed, example in enumerate(pending_embeddings, start=1):
        started = time.perf_counter()
        example["reranked"] = rerank_candidates(example["query"], example["candidates"])
        example["rerank_seconds"] = time.perf_counter() - started
        if completed % 100 == 0 or completed == len(pending_embeddings):
            print(f"Completed {completed}/{len(pending_embeddings)} embedding queries.")
    print(f"Reranked {len(embedding_examples)} embedding retrieval results.")

## Measure the difference

nDCG@K uses ViDoRe's graded qrels and rewards relevant pages near the top. Its ideal ordering uses **all judged HR pages**, including relevant pages BM25 failed to retrieve. Candidate recall is unchanged by reranking. The notebook shows only the first DISPLAY_QUERY_ROWS query rows plus full-HR means to keep output readable. Rerank time includes local model processing; it is not a controlled latency benchmark.

In [8]:
from IPython.display import Markdown, display

def ndcg_at_k(ranking, judgments, k):
    def dcg(scores):
        return sum((2**score - 1) / math.log2(rank + 1)
                   for rank, score in enumerate(scores, start=1))
    observed = [judgments.get(hit["corpus_id"], 0) for hit in ranking[:k]]
    ideal = sorted(judgments.values(), reverse=True)[:k]
    denominator = dcg(ideal)
    return dcg(observed) / denominator if denominator else 0.0

def table(rows, columns):
    def value(item):
        return (f"{item:.3f}" if isinstance(item, float) else str(item)).replace("|", "\\|")
    lines = ["| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(value(row[col]) for col in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

results = []
for example in examples:
    before = ndcg_at_k(example["candidates"], example["relevance"], TOP_K)
    after = ndcg_at_k(example["reranked"], example["relevance"], TOP_K)
    relevant_ids = set(example["relevance"])
    retrieved_ids = {hit["corpus_id"] for hit in example["candidates"]}
    results.append({"query_id": example["query_id"],
                    "No Reranker nDCG": before, RERANK_METRIC: after,
                    "delta": after - before,
                    "candidate recall": len(retrieved_ids & relevant_ids) / len(relevant_ids),
                    "rerank seconds": example["rerank_seconds"]})
metric_columns = ["query_id", "No Reranker nDCG", RERANK_METRIC,
                  "delta", "candidate recall", "rerank seconds"]
table(results[:DISPLAY_QUERY_ROWS], metric_columns)
print(f"Showing {min(DISPLAY_QUERY_ROWS, len(results))} of {len(results)} HR queries above.")
table([{"option": option, f"mean nDCG@{TOP_K}": sum(row[key] for row in results) / len(results)}
       for option, key in [("No Reranker", "No Reranker nDCG"),
                           (RERANK_LABEL, RERANK_METRIC)]],
      ["option", f"mean nDCG@{TOP_K}"])

| query_id | No Reranker nDCG | llama-nemotron-rerank-vl-1b-v2 nDCG | delta | candidate recall | rerank seconds |
| --- | --- | --- | --- | --- | --- |
| 0 | 0.000 | 0.000 | 0.000 | 0.000 | 0.532 |
| 1 | 0.000 | 0.146 | 0.146 | 0.071 | 0.216 |
| 2 | 0.316 | 0.684 | 0.369 | 0.500 | 0.217 |
| 3 | 0.146 | 0.000 | -0.146 | 0.167 | 0.216 |
| 4 | 0.307 | 0.501 | 0.195 | 1.000 | 0.213 |
| 5 | 0.642 | 0.478 | -0.165 | 0.750 | 0.213 |
| 6 | 1.000 | 1.000 | 0.000 | 1.000 | 0.217 |
| 7 | 1.000 | 1.000 | 0.000 | 1.000 | 0.218 |
| 8 | 0.237 | 0.613 | 0.376 | 0.500 | 0.213 |
| 9 | 0.854 | 0.786 | -0.068 | 0.429 | 0.217 |

Showing 10 of 1908 HR queries above.


| option | mean nDCG@5 |
| --- | --- |
| No Reranker | 0.116 |
| llama-nemotron-rerank-vl-1b-v2 | 0.187 |

In [9]:
first = examples[0]
comparison = [
    {"reranked_rank": rank, "BM25_rank": hit["original_rank"],
     "corpus_id": hit["corpus_id"],
     "relevance": first["relevance"].get(hit["corpus_id"], 0),
     "BM25_score": hit["retrieval_score"],
     "rerank_logit": hit["rerank_score"]}
    for rank, hit in enumerate(first["reranked"], start=1)
]
print(f"Detailed rank changes for query_id={first['query_id']}: {first['query']}")
table(comparison, list(comparison[0]))

Detailed rank changes for query_id=0: What impact did the shifts in temporary contract usage from 2018 to 2023 have on the difference in employment stability between mobile EU workers and citizens of the host countries?


| reranked_rank | BM25_rank | corpus_id | relevance | BM25_score | rerank_logit |
| --- | --- | --- | --- | --- | --- |
| 1 | 5 | 475 | 0 | 19.716 | -5.156 |
| 2 | 4 | 365 | 0 | 20.012 | -5.240 |
| 3 | 10 | 523 | 0 | 18.644 | -6.551 |
| 4 | 11 | 503 | 0 | 18.538 | -6.567 |
| 5 | 3 | 443 | 0 | 20.068 | -6.734 |
| 6 | 6 | 755 | 0 | 19.261 | -6.973 |
| 7 | 2 | 756 | 0 | 21.652 | -7.658 |
| 8 | 9 | 961 | 0 | 18.787 | -8.533 |
| 9 | 7 | 759 | 0 | 19.194 | -8.580 |
| 10 | 12 | 931 | 0 | 18.354 | -8.812 |
| 11 | 1 | 185 | 0 | 22.090 | -9.380 |
| 12 | 8 | 244 | 0 | 18.992 | -9.496 |

## Aggregate result across all HR queries

The final score is the mean nDCG across **all** HR queries, including queries with no judged-relevant page among the BM25 candidates. The delta is reranked mean minus no-reranker mean; a positive value means the reranker improved this text-only pipeline's aggregate ranking quality.

In [10]:
if len(results) != len(queries):
    raise ValueError("Aggregate results must include every HR query.")

mean_before = sum(row["No Reranker nDCG"] for row in results) / len(results)
mean_after = sum(row[RERANK_METRIC] for row in results) / len(results)
aggregate = {
    "queries": len(results),
    f"No Reranker mean nDCG@{TOP_K}": mean_before,
    f"{RERANK_LABEL} mean nDCG@{TOP_K}": mean_after,
    f"delta mean nDCG@{TOP_K}": mean_after - mean_before,
}
table([aggregate], list(aggregate))

| queries | No Reranker mean nDCG@5 | llama-nemotron-rerank-vl-1b-v2 mean nDCG@5 | delta mean nDCG@5 |
| --- | --- | --- | --- |
| 1908 | 0.116 | 0.187 | 0.071 |

## Optional embedding retrieval: aggregate score and delta

When enabled, this table includes **all HR queries** for both candidate sources. Each reranked row's delta is measured against its own no-reranker candidate order; candidate recall stays constant within that pair. Differences between BM25 and embedding rows include a change in retrieved candidates, so they do not isolate the reranker's effect.

In [11]:
if RUN_EMBEDDING_RETRIEVAL:
    if len(embedding_examples) != len(queries):
        raise ValueError("Embedding results must include every HR query.")
    if any("reranked" not in example for example in embedding_examples):
        raise ValueError("Rerank the embedding candidates before measuring quality.")
    embedding_results = []
    for example in embedding_examples:
        judgments = example["relevance"]
        retrieved_ids = {hit["corpus_id"] for hit in example["candidates"]}
        embedding_results.append({
            "retrieval nDCG": ndcg_at_k(example["candidates"], judgments, TOP_K),
            "reranked nDCG": ndcg_at_k(example["reranked"], judgments, TOP_K),
            "candidate recall": len(retrieved_ids & set(judgments)) / len(judgments),
        })
    mean_embedding_before = sum(row["retrieval nDCG"] for row in embedding_results) / len(queries)
    mean_embedding_after = sum(row["reranked nDCG"] for row in embedding_results) / len(queries)
    mean_bm25_recall = sum(row["candidate recall"] for row in results) / len(queries)
    mean_embedding_recall = sum(row["candidate recall"] for row in embedding_results) / len(queries)
    comparison_rows = [
        {"candidate source": "BM25", "reranker": "No",
         "mean nDCG": mean_before, "delta vs source": 0.0,
         "mean candidate recall": mean_bm25_recall},
        {"candidate source": "BM25", "reranker": RERANK_LABEL,
         "mean nDCG": mean_after, "delta vs source": mean_after - mean_before,
         "mean candidate recall": mean_bm25_recall},
        {"candidate source": "Nemotron 3 Embed", "reranker": "No",
         "mean nDCG": mean_embedding_before, "delta vs source": 0.0,
         "mean candidate recall": mean_embedding_recall},
        {"candidate source": "Nemotron 3 Embed", "reranker": RERANK_LABEL,
         "mean nDCG": mean_embedding_after,
         "delta vs source": mean_embedding_after - mean_embedding_before,
         "mean candidate recall": mean_embedding_recall},
    ]
    print(f"All {len(queries)} HR queries; nDCG@{TOP_K} and recall at candidate depth {CANDIDATES_PER_QUERY}.")
    table(comparison_rows, list(comparison_rows[0]))
else:
    print("Optional embedding comparison skipped.")

Optional embedding comparison skipped.


## Interpret the result

- **Fixed candidates:** Within each BM25 or embedding pair, the reranker changes order, not recall. Increase retrieval depth if positives are absent.
- **Quality versus cost:** More candidates can improve opportunities for reranking but increase inference and latency. The optional embedding path also adds a model download, embedding inference, and another full reranking pass.
- **Text-only limitation:** BM25, embeddings, and reranking use extracted markdown, not page images. Charts, layout, or visual evidence absent from that text are unavailable to these paths. These results are not a visual-document benchmark score.
- **Model fit:** Rerankers can differ in training data and supported modalities. Measure the configured model's text-only quality on your own data rather than assuming visual-document results transfer.
- **Operational fit:** Local inference keeps query and document text in your environment, but requires significant model storage and a BF16-capable GPU with enough memory. Check source-document licenses before redistributing outputs.
- **Model comparison:** The within-source deltas measure reranking; differences across BM25 and embedding rows also reflect different candidate sets and retrieval input lengths.